In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from keras import regularizers, layers
import matplotlib.pyplot as plt
import time
import numpy as np
import os
from IPython import display
import shutil
import cv2

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
(x_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()

In [ ]:
def preprocess_dataset(images, labels, num_classes):
    # Normalize images to [-1, 1]
    images = tf.cast(images, tf.float32)
    images = (images - 127.5) / 127.5
    labels = tf.reshape(labels, shape=(1,))
    return images, labels

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.map(lambda x, y: preprocess_dataset(x, y, num_classes=10)).batch(128)
train_dataset = train_dataset.cache().prefetch(buffer_size=tf.data.experimental.AUTOTUNE).shuffle(buffer_size=train_dataset.cardinality(), reshuffle_each_iteration=True)

In [ ]:
def make_discriminator_model(in_shape=(28, 28, 1), n_classes=10):
    in_image = layers.Input(shape=in_shape)

    # Embed class IDs into an image-sized conditioning map.
    in_label = layers.Input(shape=(1,))
    label = layers.Embedding(n_classes, 50)(in_label)

    y = layers.Dense(in_shape[0] * in_shape[1], use_bias=False)(label)
    y = layers.Reshape((in_shape[0], in_shape[1], 1))(y)

    merge = layers.Concatenate()([in_image, y])

    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(merge)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02)(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02)(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(32, (5, 5), strides=(2, 2), padding='same', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02)(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model([in_image, in_label], outputs)
    return model


In [ ]:
def make_generator_model():
    noise_dim = 100
    num_classes = 10

    noise_input = layers.Input(shape=(noise_dim,))
    
    label_input = layers.Input(shape=(1,))
    label_embedding = layers.Embedding(num_classes, 50)(label_input)
    label = layers.Dense(7 * 7)(label_embedding)
    label = layers.Reshape((7,7, 1))(label)
    
    gen = layers.Dense(7*7*128)(noise_input)
    gen = layers.LeakyReLU(alpha=0.2)(gen)
    gen = layers.Reshape((7, 7, 128))(gen)

    merge = layers.Concatenate()([gen, label])

    x = layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False, kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(merge)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False, kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02)(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(32, (5, 5), strides=(2, 2), padding='same', use_bias=False, kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02)(x)
    x = layers.ReLU()(x)
    
    image_output = layers.Conv2DTranspose(1, (5, 5), strides=(1, 1), padding='same', activation='tanh', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    
    generator = keras.Model([noise_input, label_input], image_output)
    return generator

In [ ]:
generator = make_generator_model()
discriminator = make_discriminator_model()

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=False)

In [ ]:
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

In [ ]:
def generator_loss(fake_output, true_labels):
    adv_loss = tf.keras.losses.BinaryCrossentropy(from_logits=False)(tf.ones_like(fake_output), fake_output)
    
    return adv_loss

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

In [ ]:
EPOCHS = 100
noise_dim = 100
num_examples_to_generate = 16

seed = tf.random.normal([num_examples_to_generate, noise_dim])

In [ ]:
@tf.function
def train_step(images, labels):
    batch_size = tf.shape(labels)[0]
    noise = tf.random.normal([batch_size, 100])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        generated_images = generator([noise, labels], training=True)

        real_output = discriminator([images, labels], training=True)
        fake_output = discriminator([generated_images, labels], training=True)

        gen_loss = generator_loss(fake_output, labels)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))

    batch_size = tf.shape(labels)[0]
    noise = tf.random.normal([batch_size, 100])
    
    # Update the generator twice per discriminator step.
    with tf.GradientTape() as second_gen_tape:
        generated_images = generator([noise, labels], training=True)
        fake_output = discriminator([generated_images, labels], training=True)

        second_gen_loss = generator_loss(fake_output, labels)

    second_gradients_of_generator = second_gen_tape.gradient(second_gen_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(second_gradients_of_generator, generator.trainable_variables))

In [ ]:
def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()

        for image_batch, label_batch in dataset:
            train_step(image_batch, label_batch)

        batch_size = 16
        test_labels = generate_random_labels(batch_size)


        display.clear_output(wait=True)
        generate_and_save_images(generator,
                                 epoch + 1,
                                 seed,test_labels)

        if epoch == 100:
            checkpoint.save(file_prefix=checkpoint_prefix)

        print('Time for epoch {} is {} sec'.format(epoch + 1, time.time() - start))

    display.clear_output(wait=True)
    generate_and_save_images(generator,
                             epochs,
                             seed,test_labels)

In [ ]:
def generate_random_labels(batch_size, num_classes=10):
    labels = tf.random.uniform(
        shape=(batch_size, 1),
        minval=0,
        maxval=num_classes,  # maxval is exclusive
        dtype=tf.int32
    )
    return labels

In [ ]:
def generate_and_save_images(model, epoch, test_input, test_labels):

    predictions = model([test_input, test_labels], training=False)

    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')

    plt.show()


In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
train(train_dataset, EPOCHS)

In [ ]:
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

In [ ]:
def save_images_with_labels(generator, num_images, folder, batch_size=1000):
    os.makedirs(folder, exist_ok=True)

    for batch_start in range(0, num_images, batch_size):
        current_batch_size = min(batch_size, num_images - batch_start)

        noise = np.random.normal(0, 1, (current_batch_size, 100))
        noise = tf.convert_to_tensor(noise, dtype=tf.float32)
        
        test_labels = generate_random_labels(current_batch_size)
        
        generated_images = generator([noise, test_labels])
        
        predicted_labels = test_labels.numpy()

        for i, (image, label) in enumerate(zip(generated_images.numpy(), predicted_labels)):

            label_name = int(label)

            # Clip generated pixels to the display range [0, 1].
            image = np.clip(image, 0, 1)

            image = np.repeat(image, 3, axis=-1)

            image_index = batch_start + i
            save_path = f'{folder}/generated_image_{image_index}_label_{label_name}.png'
            plt.imsave(save_path, image.reshape(28, 28, 3), cmap='gray')

            shutil.move(save_path, os.path.join(folder, os.path.basename(save_path)))


In [ ]:
save_dir = "generated_images"
save_images_with_labels(generator, num_images=100000, folder=save_dir)

In [ ]:
image_dir = '/kaggle/working/generated_images'
parent_dir = 'generated_images_with_images'

os.makedirs(parent_dir, exist_ok=True)

for image_file in os.listdir(image_dir):
    if image_file.endswith('.png'):
            # Labels follow "label_" in generated filenames.
        label = image_file.split('_label_')[-1].split('_')[0]

        label_dir = os.path.join(parent_dir, label)
        os.makedirs(label_dir, exist_ok=True)

        source_path = os.path.join(image_dir, image_file)
        target_path = os.path.join(label_dir, image_file)

        if os.path.exists(target_path):
            base, ext = os.path.splitext(image_file)
            target_path = os.path.join(label_dir, f"{base}_duplicate{ext}")

        shutil.move(source_path, target_path)
